In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

train = pd.read_csv('/kaggle/input/datasets/mauron67/alfa-mfti-data-2/train_apps.csv')
test = pd.read_csv('/kaggle/input/datasets/mauron67/alfa-mfti-data-2/test_apps.csv')
sub = pd.read_csv('/kaggle/input/datasets/mauron67/alfa-mfti-data-2/sample_submission.csv')

test_ids = test['front_id'].copy()

train.drop('front_id', axis=1, inplace=True)
test.drop('front_id', axis=1, inplace=True)

y = train['target_value'].copy()
train.drop('target_value', axis=1, inplace=True)

for df in [train, test]:
    df['decision_day'] = pd.to_datetime(df['decision_day'])
    df['decision_month'] = df['decision_day'].dt.month.astype(int)
    df['decision_day_of_month'] = df['decision_day'].dt.day.astype(int)
    df['decision_day_of_week'] = df['decision_day'].dt.weekday.astype(int)
    df['decision_day_of_year'] = df['decision_day'].dt.dayofyear.astype(int)
    df.drop('decision_day', axis=1, inplace=True)


train['rate_diff'] = train['offered_rate'] - train['cb_rate']
test['rate_diff'] = test['offered_rate'] - test['cb_rate']

train['overdraft_range'] = train['overdraft_limit_max'] - train['overdraft_limit_min']
test['overdraft_range'] = test['overdraft_limit_max'] - test['overdraft_limit_min']

train['deb_ul_diff_90_30'] = train['sum_deb_ul_90'] - train['sum_deb_ul_30']
test['deb_ul_diff_90_30'] = test['sum_deb_ul_90'] - test['sum_deb_ul_30']

train['cnt_deb_ul_ip_diff'] = train['cnt_deb_ul_ip_90'] - train['cnt_deb_ul_ip_30']
test['cnt_deb_ul_ip_diff'] = test['cnt_deb_ul_ip_90'] - test['cnt_deb_ul_ip_30']


cat_features = ['db_group_last', 'fl_adminarea']

for col in cat_features:
    train[col] = train[col].astype(str)
    test[col] = test[col].astype(str)


params = {
    'iterations': 3000,
    'depth': 6,
    'learning_rate': 0.03,
    'eval_metric': 'AUC',
    'early_stopping_rounds': 150,
    'task_type': 'GPU',
    'devices': '0',            # одна T4
    'random_seed': 42,
    'verbose': 100,
    'allow_writing_files': False,
}


n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
test_preds = np.zeros((n_splits, len(test)))
oof_preds = np.zeros(len(train))
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train, y)):
    X_tr, X_val = train.iloc[train_idx], train.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(**params)
    model.fit(
        X_tr, y_tr,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        use_best_model=True,
        verbose=100
    )


    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, oof_preds[val_idx])
    scores.append(score)
    print(f'Fold {fold+1} ROC-AUC: {score:.5f}')


    test_preds[fold, :] = model.predict_proba(test)[:, 1]

print(f'\nСредний OOF ROC-AUC: {np.mean(scores):.5f} ± {np.std(scores):.5f}')


final_test_preds = test_preds.mean(axis=0)


submission = pd.DataFrame({
    'front_id': test_ids,
    'target_value': final_test_preds
})
submission.to_csv('submission.csv', index=False)
print('Сабмит сохранён в submission.csv')

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7450942	best: 0.7450942 (0)	total: 248ms	remaining: 12m 23s
100:	test: 0.8100347	best: 0.8100347 (100)	total: 2.24s	remaining: 1m 4s
200:	test: 0.8170977	best: 0.8170977 (200)	total: 4.09s	remaining: 57s
300:	test: 0.8204609	best: 0.8204609 (300)	total: 5.99s	remaining: 53.7s
400:	test: 0.8225650	best: 0.8225748 (399)	total: 7.83s	remaining: 50.8s
500:	test: 0.8236722	best: 0.8236722 (500)	total: 9.71s	remaining: 48.4s
600:	test: 0.8249386	best: 0.8249411 (599)	total: 11.5s	remaining: 46s
700:	test: 0.8257498	best: 0.8257498 (700)	total: 13.4s	remaining: 43.9s
800:	test: 0.8263267	best: 0.8263267 (800)	total: 15.2s	remaining: 41.8s
900:	test: 0.8269368	best: 0.8269368 (900)	total: 17s	remaining: 39.7s
1000:	test: 0.8273913	best: 0.8273913 (1000)	total: 18.9s	remaining: 37.7s
1100:	test: 0.8277273	best: 0.8277635 (1097)	total: 20.7s	remaining: 35.8s
1200:	test: 0.8280479	best: 0.8280744 (1193)	total: 22.6s	remaining: 33.9s
1300:	test: 0.8283316	best: 0.8283504 (1298)	total: 2

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7029113	best: 0.7029113 (0)	total: 19ms	remaining: 57.1s
100:	test: 0.8167633	best: 0.8167633 (100)	total: 1.89s	remaining: 54.1s
200:	test: 0.8228619	best: 0.8228619 (200)	total: 3.74s	remaining: 52.1s
300:	test: 0.8256525	best: 0.8256525 (300)	total: 5.6s	remaining: 50.2s
400:	test: 0.8271945	best: 0.8271945 (400)	total: 7.46s	remaining: 48.4s
500:	test: 0.8285118	best: 0.8285118 (500)	total: 9.27s	remaining: 46.2s
600:	test: 0.8295164	best: 0.8295164 (600)	total: 11.1s	remaining: 44.3s
700:	test: 0.8300398	best: 0.8300398 (700)	total: 12.9s	remaining: 42.4s
800:	test: 0.8305568	best: 0.8305604 (795)	total: 14.8s	remaining: 40.6s
900:	test: 0.8311761	best: 0.8311907 (898)	total: 16.6s	remaining: 38.7s
1000:	test: 0.8316830	best: 0.8316830 (999)	total: 18.4s	remaining: 36.8s
1100:	test: 0.8322044	best: 0.8322093 (1098)	total: 20.3s	remaining: 34.9s
1200:	test: 0.8327278	best: 0.8327290 (1199)	total: 22.1s	remaining: 33.1s
1300:	test: 0.8329325	best: 0.8329341 (1287)	total: 

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7032978	best: 0.7032978 (0)	total: 20.9ms	remaining: 1m 2s
100:	test: 0.8258203	best: 0.8258203 (100)	total: 1.91s	remaining: 54.8s
200:	test: 0.8318684	best: 0.8318684 (200)	total: 3.79s	remaining: 52.8s
300:	test: 0.8354995	best: 0.8354995 (300)	total: 5.67s	remaining: 50.9s
400:	test: 0.8373758	best: 0.8374052 (399)	total: 7.53s	remaining: 48.8s
500:	test: 0.8382766	best: 0.8382798 (499)	total: 9.36s	remaining: 46.7s
600:	test: 0.8390588	best: 0.8390588 (599)	total: 11.2s	remaining: 44.5s
700:	test: 0.8397447	best: 0.8397808 (697)	total: 13s	remaining: 42.5s
800:	test: 0.8402020	best: 0.8402680 (794)	total: 14.7s	remaining: 40.5s
900:	test: 0.8407035	best: 0.8407456 (897)	total: 16.5s	remaining: 38.5s
1000:	test: 0.8408791	best: 0.8409771 (980)	total: 18.4s	remaining: 36.7s
1100:	test: 0.8412043	best: 0.8412697 (1081)	total: 20.2s	remaining: 34.8s
1200:	test: 0.8413886	best: 0.8413965 (1198)	total: 22s	remaining: 32.9s
1300:	test: 0.8416494	best: 0.8416662 (1298)	total: 2

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7535046	best: 0.7535046 (0)	total: 19ms	remaining: 57s
100:	test: 0.8191545	best: 0.8191545 (100)	total: 1.83s	remaining: 52.5s
200:	test: 0.8264130	best: 0.8264314 (198)	total: 3.6s	remaining: 50.2s
300:	test: 0.8296695	best: 0.8296695 (300)	total: 5.4s	remaining: 48.4s
400:	test: 0.8312856	best: 0.8312856 (400)	total: 7.16s	remaining: 46.4s
500:	test: 0.8324330	best: 0.8324368 (498)	total: 8.91s	remaining: 44.4s
600:	test: 0.8335996	best: 0.8335996 (600)	total: 10.7s	remaining: 42.6s
700:	test: 0.8341431	best: 0.8341483 (685)	total: 12.4s	remaining: 40.8s
800:	test: 0.8348875	best: 0.8348875 (800)	total: 14.2s	remaining: 39.1s
900:	test: 0.8354059	best: 0.8354241 (898)	total: 16.1s	remaining: 37.4s
1000:	test: 0.8358088	best: 0.8358815 (989)	total: 17.9s	remaining: 35.7s
1100:	test: 0.8362770	best: 0.8362770 (1100)	total: 19.7s	remaining: 34s
1200:	test: 0.8366478	best: 0.8366835 (1187)	total: 21.5s	remaining: 32.2s
1300:	test: 0.8369652	best: 0.8370274 (1296)	total: 23.3s

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7334234	best: 0.7334234 (0)	total: 19.9ms	remaining: 59.8s
100:	test: 0.8119166	best: 0.8119166 (100)	total: 1.86s	remaining: 53.5s
200:	test: 0.8184886	best: 0.8184886 (200)	total: 3.71s	remaining: 51.6s
300:	test: 0.8214323	best: 0.8214323 (300)	total: 5.54s	remaining: 49.7s
400:	test: 0.8231066	best: 0.8231232 (393)	total: 7.39s	remaining: 47.9s
500:	test: 0.8245875	best: 0.8245875 (500)	total: 9.21s	remaining: 45.9s
600:	test: 0.8258184	best: 0.8258184 (600)	total: 11.1s	remaining: 44.2s
700:	test: 0.8263405	best: 0.8263405 (700)	total: 12.9s	remaining: 42.3s
800:	test: 0.8271424	best: 0.8271424 (800)	total: 14.8s	remaining: 40.6s
900:	test: 0.8277148	best: 0.8277213 (898)	total: 16.6s	remaining: 38.7s
1000:	test: 0.8283537	best: 0.8283537 (1000)	total: 18.5s	remaining: 37s
1100:	test: 0.8286932	best: 0.8287150 (1097)	total: 20.4s	remaining: 35.1s
1200:	test: 0.8289360	best: 0.8289626 (1182)	total: 22.2s	remaining: 33.3s
1300:	test: 0.8290777	best: 0.8290777 (1300)	total

In [2]:
# Округляем вероятности до 6 знаков после запятой
submission['target_value'] = submission['target_value'].round(6)

# Печатаем первые строки, чтобы убедиться в результате
print(submission.head())

# Пересохраняем файл на диск Kaggle
submission.to_csv('submission_v1.csv', index=False)
print("Готово! Числа округлены, файл сохранен.")

   front_id  target_value
0    150378      0.062966
1    194170      0.035714
2    102106      0.012613
3    256199      0.022193
4    253573      0.026503
Готово! Числа округлены, файл сохранен.


In [4]:
# 1. Считываем оригинальный шаблон организаторов
original_sub = pd.read_csv('/kaggle/input/datasets/mauron67/alfa-mfti-data-2/sample_submission.csv')

# 2. Принудительно приводим типы ID к целым числам в обоих датафреймах
original_sub['front_id'] = original_sub['front_id'].astype(int)
submission['front_id'] = submission['front_id'].astype(int)

# 3. Перестраиваем наше решение строго в том порядке строк, который ждет робот
final_sub = original_sub[['front_id']].merge(submission, on='front_id', how='left')

# 4. На всякий случай заполняем пропуски средним, если что-то потерялось
final_sub['target_value'] = final_sub['target_value'].fillna(0.5).round(6)

# 5. Сохраняем итоговый файл без лишних индексов
final_sub.to_csv('/kaggle/working/submission_v2.csv', index=False)

print("Готово! Строки отсортированы строго по шаблону.")


Готово! Строки отсортированы строго по шаблону.
